In [2]:
import ee
import geemap
import os

ee.Initialize(project="albaniasat")
print("GEE connected!")

GEE connected!


In [3]:
albania = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
            .filter(ee.Filter.eq("country_na", "Albania"))

print("Albania boundary loaded")

Albania boundary loaded


In [4]:
# Clean composite with all 4 bands, summer only, cloud-free
composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B4", "B3", "B2", "B8"])
                .median())

print("Composite image ready!")
info = composite.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

Composite image ready!
Bands: ['B4', 'B3', 'B2', 'B8']


In [5]:
# Load CORINE Land Cover map
corine = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")

# Clip it to Albania only
corine_albania = corine.clip(albania.geometry())

# Check what classes exist in Albania
values = corine_albania.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=albania.geometry(),
    scale=100,
    maxPixels=1e9
).getInfo()

print("Land cover classes found in Albania:")
for code, count in sorted(values["landcover"].items(), key=lambda x: -x[1]):
    print(f"  Class {code}: {int(count)} pixels")

Land cover classes found in Albania:
  Class 311: 577016 pixels
  Class 324: 394805 pixels
  Class 321: 308616 pixels
  Class 323: 285155 pixels
  Class 243: 248968 pixels
  Class 211: 203390 pixels
  Class 242: 195793 pixels
  Class 333: 143038 pixels
  Class 231: 77403 pixels
  Class 312: 74283 pixels
  Class 112: 65517 pixels
  Class 313: 59311 pixels
  Class 223: 48901 pixels
  Class 512: 44968 pixels
  Class 322: 23377 pixels
  Class 332: 19883 pixels
  Class 331: 19643 pixels
  Class 222: 13859 pixels
  Class 212: 12825 pixels
  Class 511: 8382 pixels
  Class 334: 5845 pixels
  Class 421: 5200 pixels
  Class 521: 4639 pixels
  Class 121: 4247 pixels
  Class 411: 4234 pixels
  Class 523: 3659 pixels
  Class 221: 3599 pixels
  Class 131: 3242 pixels
  Class 335: 2293 pixels
  Class 133: 1159 pixels
  Class 124: 1062 pixels
  Class 422: 947 pixels
  Class 111: 663 pixels
  Class 122: 486 pixels
  Class 142: 353 pixels
  Class 141: 254 pixels
  Class 241: 237 pixels
  Class 123: 159 

## CORINE Land Cover Classes Found in Albania

| Code | Class Name                     | Pixels  |
|------|--------------------------------|---------|
| 311  | Broad-leaved forest            | 577,016 |
| 324  | Transitional woodland/shrub    | 394,805 |
| 321  | Natural grassland              | 308,616 |
| 323  | Sclerophyllous vegetation      | 285,155 |
| 243  | Agriculture + natural veg.     | 248,968 |
| 211  | Non-irrigated arable land      | 203,390 |
| 242  | Complex cultivation patterns   | 195,793 |
| 333  | Sparsely vegetated areas       | 143,038 |
| 231  | Pastures                       | 77,403  |
| 312  | Coniferous forest              | 74,283  |
| 112  | Discontinuous urban fabric     | 65,517  |
| 313  | Mixed forest                   | 59,311  |
| 223  | Olive groves                   | 48,901  |
| 512  | Water bodies                   | 44,968  |

## AlbaniaSAT Class Groupings

| AlbaniaSAT Class | CORINE Codes  |
|------------------|---------------|
| Broad-leaved Forest | 311        |
| Coniferous Forest   | 312        |
| Shrubland           | 323, 324   |
| Agricultural        | 211, 242, 243 |
| Grassland           | 231, 321   |
| Olive Groves        | 223        |
| Urban               | 112        |
| Water               | 512        |

1. Spectral Honesty

Broad-leaved and coniferous forests reflect light differently, broad-leaved trees have wide flat leaves, coniferous have thin needles. Sentinel-2 can clearly see this difference, especially in the NIR band. Merging them into one class would be throwing away real signal.

2. A More Meaningful Benchmark

Finer-grained classes make the dataset more detailed than EuroSAT, not less. A model that can tell broad-leaved from coniferous forest is a more capable model than one that just says "forest." That's a better result to publish.

In [6]:
# Define 8 classes with their CORINE codes
classes = {
    "Broad-leaved Forest": [311],
    "Coniferous Forest": [312],
    "Shrubland": [323, 324],
    "Agricultural": [211, 242, 243],
    "Grassland": [231, 321],
    "Olive Groves": [223],
    "Urban": [112],
    "Water": [512],
}

N_PATCHES = 500
all_samples = {}

for class_name, codes in classes.items():
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))

    points = mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=42,
        geometries=True
    )

    count = points.size().getInfo()
    all_samples[class_name] = points
    print(f"{class_name}: {count} sample points found")

Broad-leaved Forest: 500 sample points found
Coniferous Forest: 500 sample points found
Shrubland: 500 sample points found
Agricultural: 500 sample points found
Grassland: 500 sample points found
Olive Groves: 500 sample points found
Urban: 500 sample points found
Water: 500 sample points found


In [24]:
PATCH_SIZE = 32  # 32 pixels each direction = 64x64 total

patches = composite.select(["B4", "B3", "B2", "B8"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name, points in all_samples.items():
    clean_name = class_name.replace(" ", "_")
    
    samples = patches.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"patches_{clean_name}",
        folder="AlbaniaSAT_v3",
        fileNamePrefix=f"{clean_name}",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Broad-leaved Forest
Queued: Coniferous Forest
Queued: Shrubland
Queued: Agricultural
Queued: Grassland
Queued: Olive Groves
Queued: Urban
Queued: Water
All 8 tasks queued!


## v2 — Expanded Dataset (1000 patches per class)
Goal: increase from 500 to 1000 patches per class to reduce overfitting.

In [7]:
# Load existing v1 sample points and buffer them
# This ensures new points are at least 640m away from existing ones

N_PATCHES_V2 = 500  # 500 new + 500 existing = 1000 total per class
all_samples_v2 = {}

for class_name, codes in classes.items():
    # Build class mask
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))
    
    # Get existing v1 points for this class
    existing = all_samples[class_name]
    
    # Create 640m exclusion zone around existing points
    exclusion_zone = existing.geometry().buffer(640)
    
    # Convert exclusion zone to a mask
    exclusion_mask = ee.Image.constant(1).clip(albania.geometry()).paint(
        featureCollection=ee.FeatureCollection([ee.Feature(exclusion_zone)]),
        color=0
    )
    
    # Apply exclusion to class mask
    clean_mask = mask.updateMask(exclusion_mask)
    
    # Sample from remaining area
    points = clean_mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES_V2,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=123,
        geometries=True
    )
    
    count = points.size().getInfo()
    all_samples_v2[class_name] = points
    print(f"{class_name}: {count} new points found")

Broad-leaved Forest: 500 new points found
Coniferous Forest: 500 new points found
Shrubland: 500 new points found
Agricultural: 500 new points found
Grassland: 500 new points found
Olive Groves: 500 new points found
Urban: 500 new points found
Water: 500 new points found


In [8]:
PATCH_SIZE = 32

patches_v2 = composite.select(["B4", "B3", "B2", "B8"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name, points in all_samples_v2.items():
    clean_name = class_name.replace(" ", "_")
    
    samples = patches_v2.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"patches_v2_{clean_name}",
        folder="AlbaniaSAT_v4",
        fileNamePrefix=f"{clean_name}_v2",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Broad-leaved Forest
Queued: Coniferous Forest
Queued: Shrubland
Queued: Agricultural
Queued: Grassland
Queued: Olive Groves
Queued: Urban
Queued: Water
All 8 tasks queued!


In [ ]:
import json
import numpy as np
import os

RAW_V1 = "../data/AlbaniaSAT/raw"
RAW_V2 = "../data/AlbaniaSAT/raw_v2"
OUTPUT_PATH = "../data/AlbaniaSAT/processed_v2"
os.makedirs(OUTPUT_PATH, exist_ok=True)

BANDS = ["B4", "B3", "B2", "B8"]

class_names = [
    "Broad-leaved_Forest",
    "Coniferous_Forest",
    "Shrubland",
    "Agricultural",
    "Grassland",
    "Olive_Groves",
    "Urban",
    "Water"
]

all_patches = []
all_labels = []

for label_idx, class_name in enumerate(class_names):
    # Load v1
    with open(os.path.join(RAW_V1, f"{class_name}.geojson")) as f:
        data_v1 = json.load(f)
    
    # Load v2
    with open(os.path.join(RAW_V2, f"{class_name}_v2.geojson")) as f:
        data_v2 = json.load(f)
    
    # Merge features
    all_features = data_v1["features"] + data_v2["features"]
    
    patches = []
    for feature in all_features:
        props = feature["properties"]
        bands = np.array([props[b] for b in BANDS], dtype=np.float32)
        bands = bands[:, :64, :64]
        patches.append(bands)
    
    patches = np.array(patches)
    labels = np.full(len(patches), label_idx, dtype=np.int64)
    
    all_patches.append(patches)
    all_labels.append(labels)
    print(f"{class_name}: {len(patches)} patches")

X = np.concatenate(all_patches, axis=0)
y = np.concatenate(all_labels, axis=0)

print(f"\nFinal dataset: {X.shape} patches, {y.shape} labels")

np.save(os.path.join(OUTPUT_PATH, "patches.npy"), X)
np.save(os.path.join(OUTPUT_PATH, "labels.npy"), y)
print("Saved to processed_v2!")